# Silver Layer Preparation: Bronze to Silver Transformation

## Purpose

This notebook focuses on reading raw CSV files from the **Bronze volume** and transforming them into clean, business-ready **Silver Delta tables**.

## What This Notebook Does

**Input**: Raw CSV files stored in `/Volumes/workspace/bronze/raw_data/`
* dim_customer.csv
* dim_store.csv
* dim_sku.csv
* dim_product.csv
* dim_category.csv
* dim_department.csv
* fact_transaction.csv
* fact_line_item.csv

**Transformation**: 
* Data quality checks and cleansing
* Joining related dimension tables to build hierarchies
* Denormalizing dimensions into fact tables
* Standardizing formats and data types

**Output**: Silver Delta tables in `workspace.silver` schema
* silver.customer_master
* silver.store_master
* silver.sku_master
* silver.transactions
* silver.line_items
* silver.dim_date

## Silver Layer Characteristics

The Silver layer represents **cleaned and enriched data** that is:
* Validated and deduplicated
* Enriched with business context through joins
* Optimized for analytical queries
* Stored as Delta tables for ACID compliance
* Ready for downstream consumption by analysts and data scientists

---

In [0]:

# STEP 1: Identify Available CSV Files in Bronze Volume
# Purpose:
# Before creating Silver tables, we need to see what raw CSV
# files are available in the Bronze volume's raw_data folder.
#
# This helps us understand:
# • What source files we have to work with
# • File naming conventions
# • Which dimensions and facts are available
# ===========================================================

# List all files in the Bronze raw_data volume
files = dbutils.fs.ls('/Volumes/workspace/bronze/raw_data/')

print("Available CSV files in Bronze volume:")
print("=" * 50)
for file in files:
    print(f"  • {file.name}")
print("=" * 50)
print(f"\nTotal files: {len(files)}")

Available CSV files in Bronze volume:
  • dim_category.csv
  • dim_customer.csv
  • dim_department.csv
  • dim_product.csv
  • dim_sku.csv
  • dim_store.csv
  • fact_line_item.csv
  • fact_transaction.csv

Total files: 8


## Creating silver.sku_master

**Purpose**: Create a complete product dimension with full hierarchy

**Source Files**:
* `/Volumes/workspace/bronze/raw_data/dim_sku.csv` (SKU-level attributes)
* `/Volumes/workspace/bronze/raw_data/dim_product.csv` (Product and brand details)
* `/Volumes/workspace/bronze/raw_data/dim_category.csv` (Category information)
* `/Volumes/workspace/bronze/raw_data/dim_department.csv` (Department information)

**What we're doing**:
* Reading 4 CSV files from Bronze volume
* Joining them to create complete product hierarchy
* Building: SKU → Product → Category → Department
* Enriching with brand, pricing, and packaging details

**Why this matters**:
* Enables product hierarchy analysis
* Supports category and department reporting
* Provides brand and private label analysis
* Single denormalized table avoids multiple joins downstream

In [0]:
# ===========================================================
# Preview Product-Related CSV Files
# ===========================================================
# View sample data from all 4 CSV files we'll join
# to understand the product hierarchy
# ===========================================================

# Read all 4 product-related CSV files
df_sku = spark.read.csv('/Volumes/workspace/bronze/raw_data/dim_sku.csv', header=True, inferSchema=True)
df_product = spark.read.csv('/Volumes/workspace/bronze/raw_data/dim_product.csv', header=True, inferSchema=True)
df_category = spark.read.csv('/Volumes/workspace/bronze/raw_data/dim_category.csv', header=True, inferSchema=True)
df_department = spark.read.csv('/Volumes/workspace/bronze/raw_data/dim_department.csv', header=True, inferSchema=True)

print("1. SKU Level (dim_sku.csv):")
display(df_sku.limit(3))

print("\n2. Product Level (dim_product.csv):")
display(df_product.limit(3))

print("\n3. Category Level (dim_category.csv):")
display(df_category.limit(3))

print("\n4. Department Level (dim_department.csv):")
display(df_department.limit(3))

1. SKU Level (dim_sku.csv):


sku_id,sku_name,product_id,pack_size,pack_quantity,mrp
SKU-0001,India Gate Basmati 1kg,PRD-0001,1kg,1.0,197.0
SKU-0002,India Gate Basmati 5kg,PRD-0001,5kg,5.0,657.0
SKU-0003,India Gate Basmati 10kg,PRD-0001,10kg,10.0,1182.0



2. Product Level (dim_product.csv):


product_id,product_name,category_id,category_name,brand,is_private_label
PRD-0001,India Gate Basmati,CAT-01-01,Rice & Grains,India Gate,false
PRD-0002,Daawat Biryani Rice,CAT-01-01,Rice & Grains,Daawat,false
PRD-0003,Kohinoor Silver Range,CAT-01-01,Rice & Grains,Kohinoor,false



3. Category Level (dim_category.csv):


category_id,category_name,department_id
CAT-01-01,Rice & Grains,DEPT-01
CAT-01-02,Cooking Oil & Ghee,DEPT-01
CAT-01-03,Spices & Masalas,DEPT-01



4. Department Level (dim_department.csv):


department_id,department_name
DEPT-01,Grocery & Staples
DEPT-02,Dairy & Frozen
DEPT-03,Personal Care


In [0]:
# ===========================================================
# CREATE TABLE: silver.sku_master
# ===========================================================
# Purpose:
# Create a complete product dimension by joining SKU, Product,
# Category, and Department CSV files into a single denormalized table.
#
# Grain:
# One Row = One Unique SKU
#
# Source Files:
# 1. dim_sku.csv (SKU attributes)
# 2. dim_product.csv (Product/Brand attributes)
# 3. dim_category.csv (Category attributes)
# 4. dim_department.csv (Department attributes)
#
# Product Hierarchy:
# SKU (Lowest) → Product → Category → Department (Highest)
#
# Example Hierarchy:
# Department: Grocery
#   └─ Category: Beverages
#       └─ Product: Coca Cola 2L
#           └─ SKU: Coca Cola 2L - 6 Pack
#
# Join Strategy:
# INNER JOINs ensure we only include SKUs with complete
# hierarchy information (SKU → Product → Category → Department)
#
# Downstream Use Cases:
# • Product performance analysis
# • Category sales reporting
# • Department-level metrics
# • Brand analysis
# • Private label vs branded product comparison
# • Pricing strategy analysis (MRP)
#
# Why create this table?
# Analysts need product hierarchy for every product analysis.
# This single table eliminates the need to join 4 CSV files
# repeatedly and provides all product context in one place.
# ===========================================================

# Read all 4 CSV files
df_sku = spark.read.csv('/Volumes/workspace/bronze/raw_data/dim_sku.csv', header=True, inferSchema=True)
df_product = spark.read.csv('/Volumes/workspace/bronze/raw_data/dim_product.csv', header=True, inferSchema=True)
df_category = spark.read.csv('/Volumes/workspace/bronze/raw_data/dim_category.csv', header=True, inferSchema=True)
df_department = spark.read.csv('/Volumes/workspace/bronze/raw_data/dim_department.csv', header=True, inferSchema=True)

# Join tables to build product hierarchy
# Use aliases and explicit column selection to avoid duplicate columns
# Organize columns logically: IDs → Names → Attributes
df_sku_master = df_sku.alias('sku') \
    .join(df_product.alias('prod'), 'product_id', 'inner') \
    .join(df_category.alias('cat'), 'category_id', 'inner') \
    .join(df_department.alias('dept'), 'department_id', 'inner') \
    .select(
        # IDs
        'sku.sku_id',
        'sku.product_id',
        'prod.category_id',
        'cat.department_id',
        # Names
        'sku.sku_name',
        'prod.product_name',
        'prod.brand',
        'cat.category_name',
        'dept.department_name',
        # Attributes
        'sku.mrp',
        'sku.pack_size',
        'sku.pack_quantity',
        'prod.is_private_label'
    )

# Write to Silver layer as Delta table
# Use overwriteSchema to allow column reordering
df_sku_master.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.silver.sku_master')

print("Successfully created silver.sku_master")

Successfully created silver.sku_master


In [0]:
# ===========================================================
# Verify Created Table
# ===========================================================
# Check sku_master table and validate product hierarchy
# ===========================================================

# Read the created table
df_sku_check = spark.table('workspace.silver.sku_master')

# Show statistics
print("Table Statistics:")
print(f"Total SKUs: {df_sku_check.count():,}")
print(f"Unique SKU IDs: {df_sku_check.select('sku_id').distinct().count():,}")
print(f"Unique Products: {df_sku_check.select('product_id').distinct().count():,}")
print(f"Unique Brands: {df_sku_check.select('brand').distinct().count():,}")
print(f"Unique Categories: {df_sku_check.select('category_name').distinct().count():,}")
print(f"Unique Departments: {df_sku_check.select('department_name').distinct().count():,}")

# Show sample records
print("\nSample records from silver.sku_master:")
display(df_sku_check.limit(10))

Table Statistics:
Total SKUs: 1,418
Unique SKU IDs: 1,418
Unique Products: 591
Unique Brands: 92
Unique Categories: 36
Unique Departments: 7

Sample records from silver.sku_master:


sku_id,product_id,category_id,department_id,sku_name,product_name,brand,category_name,department_name,mrp,pack_size,pack_quantity,is_private_label
SKU-0001,PRD-0001,CAT-01-01,DEPT-01,India Gate Basmati 1kg,India Gate Basmati,India Gate,Rice & Grains,Grocery & Staples,197.0,1kg,1.0,false
SKU-0002,PRD-0001,CAT-01-01,DEPT-01,India Gate Basmati 5kg,India Gate Basmati,India Gate,Rice & Grains,Grocery & Staples,657.0,5kg,5.0,false
SKU-0003,PRD-0001,CAT-01-01,DEPT-01,India Gate Basmati 10kg,India Gate Basmati,India Gate,Rice & Grains,Grocery & Staples,1182.0,10kg,10.0,false
SKU-0004,PRD-0002,CAT-01-01,DEPT-01,Daawat Biryani Rice 1kg,Daawat Biryani Rice,Daawat,Rice & Grains,Grocery & Staples,230.0,1kg,1.0,false
SKU-0005,PRD-0002,CAT-01-01,DEPT-01,Daawat Biryani Rice 5kg,Daawat Biryani Rice,Daawat,Rice & Grains,Grocery & Staples,657.0,5kg,5.0,false
SKU-0006,PRD-0002,CAT-01-01,DEPT-01,Daawat Biryani Rice 10kg,Daawat Biryani Rice,Daawat,Rice & Grains,Grocery & Staples,1149.0,10kg,10.0,false
SKU-0007,PRD-0003,CAT-01-01,DEPT-01,Kohinoor Silver Range 1kg,Kohinoor Silver Range,Kohinoor,Rice & Grains,Grocery & Staples,197.0,1kg,1.0,false
SKU-0008,PRD-0003,CAT-01-01,DEPT-01,Kohinoor Silver Range 5kg,Kohinoor Silver Range,Kohinoor,Rice & Grains,Grocery & Staples,657.0,5kg,5.0,false
SKU-0009,PRD-0004,CAT-01-01,DEPT-01,FreshMart Premium Basmati 1kg,FreshMart Premium Basmati,FreshMart,Rice & Grains,Grocery & Staples,140.0,1kg,1.0,true
SKU-0010,PRD-0004,CAT-01-01,DEPT-01,FreshMart Premium Basmati 5kg,FreshMart Premium Basmati,FreshMart,Rice & Grains,Grocery & Staples,508.0,5kg,5.0,true


## Creating silver.store_master

**Purpose**: Create a clean, standardized store dimension table

**Source File**: `/Volumes/workspace/bronze/raw_data/dim_store.csv`

**What we're doing**:
* Reading raw store CSV file from Bronze volume
* Preserving all location and operational attributes
* Creating geographic hierarchy (Store → City → State → Region)
* Validating data quality

**Why this matters**:
* Enables regional and store-level performance analysis
* Provides store operational attributes (format, tier, size)
* Single source of truth for store information
* Supports geographic reporting and filtering

In [0]:
# ===========================================================
# Preview Store CSV File
# ===========================================================
# View the raw store data to understand attributes
# and geographic hierarchy
# ===========================================================

# Read the store CSV file
df_store_raw = spark.read.csv(
    '/Volumes/workspace/bronze/raw_data/dim_store.csv',
    header=True,
    inferSchema=True
)

# Show the schema
print("Schema of dim_store.csv:")
df_store_raw.printSchema()

# Show sample data
print("\nSample data from dim_store.csv:")
display(df_store_raw.limit(10))

Schema of dim_store.csv:
root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- tier: integer (nullable = true)
 |-- format: string (nullable = true)
 |-- area_sqft: integer (nullable = true)


Sample data from dim_store.csv:


store_id,store_name,city,state,region,tier,format,area_sqft
STR-BLR-001,FreshMart Bangalore,Bangalore,Karnataka,South,1,Supermarket,10178
STR-KOC-001,FreshMart Kochi,Kochi,Kerala,South,2,Supermarket,8548
STR-COI-001,FreshMart Coimbatore,Coimbatore,Tamil Nadu,South,2,Fresh,4482
STR-MYS-001,FreshMart Mysore,Mysore,Karnataka,South,2,Express,2939
STR-PUN-001,FreshMart Pune,Pune,Maharashtra,West,1,Supermarket,9866
STR-SUR-001,FreshMart Surat,Surat,Gujarat,West,2,Supermarket,9218
STR-NAG-001,FreshMart Nagpur,Nagpur,Maharashtra,West,2,Fresh,3629
STR-NSK-001,FreshMart Nashik,Nashik,Maharashtra,West,2,Express,3198
STR-NOI-001,FreshMart Noida,Noida,Uttar Pradesh,North,1,Supermarket,9403
STR-JAI-001,FreshMart Jaipur,Jaipur,Rajasthan,North,2,Supermarket,7188


In [0]:
# ===========================================================
# CREATE TABLE: silver.store_master
# ===========================================================
# Purpose:
# Create a clean store dimension table with location,
# operational attributes, and geographic hierarchy.
#
# Grain:
# One Row = One Unique Store
#
# Source File:
# /Volumes/workspace/bronze/raw_data/dim_store.csv
#
# Transformations Applied:
# 1. Read CSV with proper schema inference
# 2. Quality check: Filter out null store_id
# 3. Preserve geographic hierarchy for regional analysis
# 4. Save as Delta table
#
# Downstream Use Cases:
# • Regional sales performance analysis
# • Store format comparison (Hypermarket vs Supermarket)
# • Store tier analysis (Tier 1, 2, 3 cities)
# • Geographic expansion planning
# • Store size (area_sqft) correlation with sales
#
# Why create this table?
# Provides clean store attributes for joining with
# transactions and enables location-based analytics.
# ===========================================================

# Read store data from CSV
df_store = spark.read.csv(
    '/Volumes/workspace/bronze/raw_data/dim_store.csv',
    header=True,
    inferSchema=True
)

# Apply data quality filters
df_store_clean = df_store.filter(df_store['store_id'].isNotNull())

# Write to Silver layer as Delta table
df_store_clean.write.mode('overwrite').saveAsTable('workspace.silver.store_master')

print("Successfully created silver.store_master")

Successfully created silver.store_master


In [0]:
# ===========================================================
# Verify Created Table
# ===========================================================
# Check store_master table and validate attributes
# ===========================================================

# Read the created table
df_store_check = spark.table('workspace.silver.store_master')

# Show statistics
print("Table Statistics:")
print(f"Total Stores: {df_store_check.count():,}")
print(f"Unique Store IDs: {df_store_check.select('store_id').distinct().count():,}")

# Show sample records
print("\nSample records from silver.store_master:")
display(df_store_check.limit(10))

Table Statistics:
Total Stores: 21
Unique Store IDs: 21

Sample records from silver.store_master:


store_id,store_name,city,state,region,tier,format,area_sqft
STR-BLR-001,FreshMart Bangalore,Bangalore,Karnataka,South,1,Supermarket,10178
STR-KOC-001,FreshMart Kochi,Kochi,Kerala,South,2,Supermarket,8548
STR-COI-001,FreshMart Coimbatore,Coimbatore,Tamil Nadu,South,2,Fresh,4482
STR-MYS-001,FreshMart Mysore,Mysore,Karnataka,South,2,Express,2939
STR-PUN-001,FreshMart Pune,Pune,Maharashtra,West,1,Supermarket,9866
STR-SUR-001,FreshMart Surat,Surat,Gujarat,West,2,Supermarket,9218
STR-NAG-001,FreshMart Nagpur,Nagpur,Maharashtra,West,2,Fresh,3629
STR-NSK-001,FreshMart Nashik,Nashik,Maharashtra,West,2,Express,3198
STR-NOI-001,FreshMart Noida,Noida,Uttar Pradesh,North,1,Supermarket,9403
STR-JAI-001,FreshMart Jaipur,Jaipur,Rajasthan,North,2,Supermarket,7188


## Creating silver.customer_master

**Purpose**: Create a clean, standardized customer dimension table

**Source File**: `/Volumes/workspace/bronze/raw_data/dim_customer.csv`

**What we're doing**:
* Reading raw customer CSV file from Bronze volume
* Standardizing column names and data types
* Validating data quality (removing nulls)
* Creating a business-ready customer dimension

**Why this matters**:
* Provides a single source of truth for customer attributes
* Enables customer segmentation analysis
* Supports loyalty program reporting
* Removes need to repeatedly read CSV files

In [0]:
# ===========================================================
# Preview Customer CSV File
# ===========================================================
# Let's first look at the raw customer data from the CSV
# to understand the structure and columns
# ===========================================================

# Read the customer CSV file
df_customer_raw = spark.read.csv(
    '/Volumes/workspace/bronze/raw_data/dim_customer.csv',
    header=True,
    inferSchema=True
)

# Show the schema
print("Schema of dim_customer.csv:")
df_customer_raw.printSchema()

# Show sample data
print("\nSample data from dim_customer.csv:")
display(df_customer_raw.limit(10))

Schema of dim_customer.csv:
root
 |-- customer_id: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- locality: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)


Sample data from dim_customer.csv:


customer_id,age_group,gender,locality,loyalty_tier
CUST-BLR-0001,26-35,F,Jayanagar,Silver
CUST-BLR-0002,18-25,M,Whitefield,Silver
CUST-BLR-0003,36-45,F,Electronic City,Silver
CUST-BLR-0004,18-25,M,Electronic City,Silver
CUST-BLR-0005,26-35,F,Koramangala,Gold
CUST-BLR-0006,26-35,M,Electronic City,Gold
CUST-BLR-0007,26-35,F,Koramangala,Silver
CUST-BLR-0008,26-35,F,Koramangala,Silver
CUST-BLR-0009,26-35,M,Jayanagar,Silver
CUST-BLR-0010,60+,F,Electronic City,Silver


In [0]:
# ===========================================================
# CREATE TABLE: silver.customer_master
# ===========================================================
# Purpose:
# Create a clean customer dimension table with all demographic
# and loyalty attributes for downstream analytics.
#
# Grain:
# One Row = One Unique Customer
#
# Source File:
# /Volumes/workspace/bronze/raw_data/dim_customer.csv
#
# Transformations Applied:
# 1. Read CSV with proper schema inference
# 2. Quality check: Filter out null customer_id
# 3. Standardized table name for Silver layer
# 4. Save as Delta table for ACID properties
#
# Downstream Use Cases:
# • Customer segmentation by age_group, gender, locality
# • Loyalty tier analysis
# • Join key for transaction and line_item tables
# • Customer demographics reporting
#
# Why create this table?
# Instead of reading CSV files every time, analysts can use
# this clean, validated Delta table with fast queries.
# ===========================================================

# Read customer data from CSV
df_customer = spark.read.csv(
    '/Volumes/workspace/bronze/raw_data/dim_customer.csv',
    header=True,
    inferSchema=True
)

# Apply data quality filters and select columns
df_customer_clean = df_customer.filter(df_customer['customer_id'].isNotNull())

# Write to Silver layer as Delta table
df_customer_clean.write.mode('overwrite').saveAsTable('workspace.silver.customer_master')

print("Successfully created silver.customer_master")

Successfully created silver.customer_master


In [0]:
# ===========================================================
# Verify Created Table
# ===========================================================
# Check the newly created customer_master table
# Validate data loaded correctly and check record counts
# ===========================================================

# Read the created table
df_customer_check = spark.table('workspace.silver.customer_master')

# Show statistics
print("Table Statistics:")
print(f"Total Customers: {df_customer_check.count():,}")
print(f"Unique Customer IDs: {df_customer_check.select('customer_id').distinct().count():,}")

# Show sample records
print("\nSample records from silver.customer_master:")
display(df_customer_check.limit(10))

Table Statistics:
Total Customers: 14,359
Unique Customer IDs: 14,359

Sample records from silver.customer_master:


customer_id,age_group,gender,locality,loyalty_tier
CUST-BLR-0001,26-35,F,Jayanagar,Silver
CUST-BLR-0002,18-25,M,Whitefield,Silver
CUST-BLR-0003,36-45,F,Electronic City,Silver
CUST-BLR-0004,18-25,M,Electronic City,Silver
CUST-BLR-0005,26-35,F,Koramangala,Gold
CUST-BLR-0006,26-35,M,Electronic City,Gold
CUST-BLR-0007,26-35,F,Koramangala,Silver
CUST-BLR-0008,26-35,F,Koramangala,Silver
CUST-BLR-0009,26-35,M,Jayanagar,Silver
CUST-BLR-0010,60+,F,Electronic City,Silver


## Creating silver.transactions

**Purpose**: Create a clean transaction-level fact table

**Source Files**:
* `/Volumes/workspace/bronze/raw_data/fact_transaction.csv` (Transaction facts)

**What we're doing**:
* Reading transaction CSV file from Bronze volume
* Deduplicating transaction records
* Keeping only transaction facts (no dimension joins)
* Creating a pure fact table

**Why this matters**:
* Maintains fact table integrity - facts only, no dimensions
* Source data for silver.sales where all enrichment happens
* Clean, deduplicated transaction records
* Flexible for different analysis patterns

In [0]:
# ===========================================================
# Preview Transaction CSV File
# ===========================================================
# View the raw transaction data from the CSV
# ===========================================================

# Read the transaction CSV file
df_transaction_raw = spark.read.csv(
    '/Volumes/workspace/bronze/raw_data/fact_transaction.csv',
    header=True,
    inferSchema=True
)

# Show the schema
print("Schema of fact_transaction.csv:")
df_transaction_raw.printSchema()

# Show sample data
print("\nSample data from fact_transaction.csv:")
display(df_transaction_raw.limit(10))

# Show basic statistics
print(f"\nTotal transaction records: {df_transaction_raw.count():,}")

Schema of fact_transaction.csv:
root
 |-- transaction_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- is_loyalty: boolean (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- total_discount_amt: double (nullable = true)
 |-- n_items: integer (nullable = true)
 |-- n_line_items: integer (nullable = true)


Sample data from fact_transaction.csv:


transaction_id,store_id,transaction_date,customer_id,payment_method,is_loyalty,total_amount,total_discount_amt,n_items,n_line_items
BSK-000024CCFB1C,STR-BBS-001,2026-03-04,CUST-BBS-0270,Card,true,939.32,52.7,7,7
BSK-00006CABA65E,STR-BBS-001,2025-10-01,CUST-BBS-0016,UPI,true,1032.11,45.9,8,5
BSK-0004A2A8F43F,STR-BBS-001,2025-08-20,CUST-BBS-0057,Card,true,230.97,5.03,3,3
BSK-0005A2591BBC,STR-BBS-001,2025-07-07,ANON-BA414604,UPI,false,451.68,4.31,3,3
BSK-000958A19E9F,STR-BBS-001,2026-01-30,CUST-BBS-F0189,Cash,true,337.33,6.67,3,3
BSK-0009E8B5F163,STR-BBS-001,2025-11-16,CUST-BBS-0098,UPI,true,1258.81,61.19,8,7
BSK-000A14DDDA67,STR-BBS-001,2026-05-07,CUST-BBS-0212,Cash,true,171.07,2.93,2,2
BSK-000C8356B5B2,STR-BBS-001,2025-12-09,ANON-01A7DA1C,UPI,false,276.56,5.43,2,2
BSK-000C93A57512,STR-BBS-001,2025-10-20,CUST-BBS-F0140,UPI,true,1220.81,51.17,7,5
BSK-000DAED92C8E,STR-BBS-001,2025-10-18,ANON-DD5521E8,Cash,false,656.52,23.5,3,3



Total transaction records: 1,458,265


In [0]:
# ===========================================================
# CREATE TABLE: silver.transactions
# ===========================================================
# Purpose:
# Create a transaction-level fact table.
# Keep this as a pure fact table - no dimension joins.
# All enrichment happens in silver.sales table.
#
# Grain:
# One Row = One Transaction
#
# Source Files:
# fact_transaction.csv (main facts)
#
# Transformations Applied:
# 1. Read transaction CSV
# 2. Deduplicate by transaction_id
# 3. Filter out null transaction_id
# 4. Save as Delta table
#
# Downstream Use Cases:
# • Input for silver.sales (gets enriched with dimensions there)
# • Transaction-level fact queries
#
# Why keep it simple?
# Fact tables should contain facts only. Dimension enrichment
# happens in the consolidated sales table to avoid duplication
# and maintain flexibility.
# ===========================================================

from pyspark.sql.functions import col

# Read transaction CSV and deduplicate
# Note: Source CSV contains duplicate transaction_ids (77,494 duplicates)
# We deduplicate here to ensure one row per transaction
df_transaction = spark.read.csv(
    '/Volumes/workspace/bronze/raw_data/fact_transaction.csv',
    header=True,
    inferSchema=True
).dropDuplicates(['transaction_id'])

# Select columns (no joins, pure fact table)
df_transactions = df_transaction.select(
    # Date
    'transaction_date',
    # IDs
    'transaction_id',
    'customer_id',
    'store_id',
    # Transaction Facts
    'payment_method',
    'is_loyalty',
    'total_amount',
    'total_discount_amt',
    'n_items',
    'n_line_items'
).filter(col('transaction_id').isNotNull())

# Write to Silver layer as Delta table
# Use overwriteSchema to allow column changes
df_transactions.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.silver.transactions')

print("Successfully created silver.transactions")

Successfully created silver.transactions


In [0]:
# ===========================================================
# Verify Created Table
# ===========================================================
# Check transactions table and validate joins
# ===========================================================

# Read the created table
df_trans_check = spark.table('workspace.silver.transactions')

# Show statistics
print("Table Statistics:")
print(f"Total Transactions: {df_trans_check.count():,}")
print(f"Unique Transaction IDs: {df_trans_check.select('transaction_id').distinct().count():,}")
print(f"Unique Customers: {df_trans_check.select('customer_id').distinct().count():,}")
print(f"Unique Stores: {df_trans_check.select('store_id').distinct().count():,}")

# Show sample records
print("\nSample records from silver.transactions:")
display(df_trans_check.limit(10))

Table Statistics:
Total Transactions: 1,380,771
Unique Transaction IDs: 1,380,771
Unique Customers: 852,586
Unique Stores: 21

Sample records from silver.transactions:


transaction_date,transaction_id,customer_id,store_id,payment_method,is_loyalty,total_amount,total_discount_amt,n_items,n_line_items
2025-01-05,BSK-006DE5D1B5E1,CUST-BBS-0230,STR-BBS-001,Cash,true,1332.99,78.02,12,7
2025-07-15,BSK-010C184C028B,CUST-BBS-0058,STR-BBS-001,Cash,true,491.95,14.05,4,3
2025-11-25,BSK-01A7D2B4A8AA,ANON-2CD0C222,STR-BBS-001,UPI,false,254.52,2.47,2,2
2025-09-01,BSK-01AE6FC9869F,ANON-EAAD1171,STR-BBS-001,UPI,false,806.29,12.71,2,2
2025-08-11,BSK-02CED23A1A38,ANON-9B3EE7A1,STR-BBS-001,Cash,false,892.25,21.75,3,3
2025-05-28,BSK-032B33FAAE60,CUST-BBS-0118,STR-BBS-001,Card,true,2454.34,55.69,7,6
2025-06-23,BSK-03473A1B08CD,ANON-58F8E8C7,STR-BBS-001,Cash,false,152.89,5.1,2,2
2025-10-03,BSK-035DAA432F93,ANON-E7EC314B,STR-BBS-001,Cash,false,528.65,14.35,3,3
2026-02-13,BSK-058FB9729FEF,CUST-BBS-F0183,STR-BBS-001,Cash,true,399.37,30.62,7,5
2026-02-11,BSK-05DE455CEDFD,ANON-9DA40013,STR-BBS-001,Cash,false,666.02,7.01,2,2


**Issue Identified & Resolved:**

The source CSV file (`fact_transaction.csv`) contained 77,494 duplicate transaction_ids. Without deduplication, this will cause a join explosion, resulting in extra rows.

**Solution:** Added `.dropDuplicates(['transaction_id'])` when reading the transaction CSV in [Cell 17](#cell-e646ef83-b398-4c22-a4d1-7aa58c0079ac).

**Verification:** Row counts now match perfectly.

## Creating silver.line_items

**Purpose**: Create a clean line item-level fact table with calculated metrics

**Source Files**:
* `/Volumes/workspace/bronze/raw_data/fact_line_item.csv` (Line item facts)

**What we're doing**:
* Reading line item CSV file from Bronze volume
* Calculating discount_amount from amount and discount_pct
* Keeping only line item facts (no dimension joins)
* Creating a pure fact table with derived metrics

**Why this matters**:
* Maintains fact table integrity - facts only, no dimensions
* Adds calculated discount_amount for convenience
* Source data for silver.sales where all enrichment happens
* Clean line item records with complete metrics

In [0]:
# ===========================================================
# Preview Line Item CSV File
# ===========================================================
# View the raw line item data from the CSV
# ===========================================================

# Read the line item CSV file
df_line_item_raw = spark.read.csv(
    '/Volumes/workspace/bronze/raw_data/fact_line_item.csv',
    header=True,
    inferSchema=True
)

# Show the schema
print("Schema of fact_line_item.csv:")
df_line_item_raw.printSchema()

# Show sample data
print("\nSample data from fact_line_item.csv:")
display(df_line_item_raw.limit(10))

# Show basic statistics
print(f"\nTotal line item records: {df_line_item_raw.count():,}")

Schema of fact_line_item.csv:
root
 |-- line_item_id: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- sku_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- mrp: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- line_total: double (nullable = true)


Sample data from fact_line_item.csv:


line_item_id,transaction_id,store_id,transaction_date,sku_id,product_id,quantity,amount,mrp,discount_pct,line_total
TXN-78B6FB4169FB,BSK-421C4522CE09,STR-BBS-001,2025-01-03,SKU-0337,PRD-0135,4,42.32,43.0,0.0159,169.28
TXN-6E62F584A918,BSK-421C4522CE09,STR-BBS-001,2025-01-03,SKU-0906,PRD-0368,2,145.24,150.0,0.0317,290.48
TXN-724683E28953,BSK-421C4522CE09,STR-BBS-001,2025-01-03,SKU-0023,PRD-0009,4,434.12,439.0,0.0111,1736.48
TXN-BB7BA81728A4,BSK-421C4522CE09,STR-BBS-001,2025-01-03,SKU-0199,PRD-0078,3,202.06,204.0,0.0095,606.18
TXN-D0BCB42CD64D,BSK-421C4522CE09,STR-BBS-001,2025-01-03,SKU-0050,PRD-0019,4,440.99,449.0,0.0178,1763.96
TXN-28B5956BCEA6,BSK-421C4522CE09,STR-BBS-001,2025-01-03,SKU-0143,PRD-0056,2,111.4,113.0,0.0142,222.8
TXN-098F3B41DA19,BSK-421C4522CE09,STR-BBS-001,2025-01-03,SKU-0259,PRD-0103,2,46.62,47.0,0.008,93.24
TXN-9F90C06E5BD6,BSK-421C4522CE09,STR-BBS-001,2025-01-03,SKU-0975,PRD-0390,3,114.03,117.0,0.0254,342.09
TXN-F0FDD44BCF2D,BSK-421C4522CE09,STR-BBS-001,2025-01-03,SKU-0724,PRD-0293,2,121.46,127.0,0.0436,242.92
TXN-8C4B57354BB4,BSK-421C4522CE09,STR-BBS-001,2025-01-03,SKU-0739,PRD-0299,2,180.39,185.0,0.0249,360.78



Total line item records: 5,301,387


In [0]:
# ===========================================================
# CREATE TABLE: silver.line_items
# ===========================================================
# Purpose:
# Create a line item-level fact table with calculated fields.
# Keep this as a pure fact table - no dimension joins.
# All enrichment happens in silver.sales table.
#
# Grain:
# One Row = One Line Item (SKU sold in a transaction)
#
# Source Files:
# fact_line_item.csv (main facts)
#
# Transformations Applied:
# 1. Read line item CSV
# 2. Calculate discount_amount from amount and discount_pct
# 3. Filter out null line_item_id
# 4. Save as Delta table
#
# Downstream Use Cases:
# • Input for silver.sales (gets enriched with dimensions there)
# • Line item-level fact queries
#
# Why keep it simple?
# Fact tables should contain facts only. Dimension enrichment
# happens in the consolidated sales table to avoid duplication
# and maintain flexibility.
# ===========================================================

from pyspark.sql.functions import col, round as spark_round

# Read line item CSV
df_line_item = spark.read.csv(
    '/Volumes/workspace/bronze/raw_data/fact_line_item.csv',
    header=True,
    inferSchema=True
)

# Add calculated fields and select columns
df_line_items = df_line_item \
    .withColumn(
        'discount_amount',
        spark_round(col('amount') * (col('discount_pct') / 100), 2)
    ) \
    .select(
        # IDs
        'line_item_id',
        'transaction_id',
        'store_id',
        'sku_id',
        'product_id',
        # Date
        'transaction_date',
        # Line Item Facts
        'quantity',
        'amount',
        'mrp',
        'discount_pct',
        'discount_amount',  # Calculated field
        'line_total'
    ) \
    .filter(col('line_item_id').isNotNull())

# Write to Silver layer as Delta table
# Use overwriteSchema to allow column changes
df_line_items.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.silver.line_items')

print("Successfully created silver.line_items")

Successfully created silver.line_items


In [0]:
# ===========================================================
# Verify Created Table
# ===========================================================
# Check line_items table and validate joins
# ===========================================================

# Read the created table
df_line_check = spark.table('workspace.silver.line_items')

# Show statistics
print("Table Statistics:")
print(f"Total Line Items: {df_line_check.count():,}")
print(f"Unique Line Item IDs: {df_line_check.select('line_item_id').distinct().count():,}")
print(f"Unique SKUs: {df_line_check.select('sku_id').distinct().count():,}")
print(f"Unique Transactions: {df_line_check.select('transaction_id').distinct().count():,}")

# Show sample records
print("\nSample records from silver.line_items:")
display(df_line_check.limit(10))

Table Statistics:
Total Line Items: 5,301,387
Unique Line Item IDs: 5,301,387
Unique SKUs: 1,604
Unique Transactions: 1,380,771

Sample records from silver.line_items:


line_item_id,transaction_id,store_id,sku_id,product_id,transaction_date,quantity,amount,mrp,discount_pct,discount_amount,line_total
TXN-78B6FB4169FB,BSK-421C4522CE09,STR-BBS-001,SKU-0337,PRD-0135,2025-01-03,4,42.32,43.0,0.0159,0.01,169.28
TXN-6E62F584A918,BSK-421C4522CE09,STR-BBS-001,SKU-0906,PRD-0368,2025-01-03,2,145.24,150.0,0.0317,0.05,290.48
TXN-724683E28953,BSK-421C4522CE09,STR-BBS-001,SKU-0023,PRD-0009,2025-01-03,4,434.12,439.0,0.0111,0.05,1736.48
TXN-BB7BA81728A4,BSK-421C4522CE09,STR-BBS-001,SKU-0199,PRD-0078,2025-01-03,3,202.06,204.0,0.0095,0.02,606.18
TXN-D0BCB42CD64D,BSK-421C4522CE09,STR-BBS-001,SKU-0050,PRD-0019,2025-01-03,4,440.99,449.0,0.0178,0.08,1763.96
TXN-28B5956BCEA6,BSK-421C4522CE09,STR-BBS-001,SKU-0143,PRD-0056,2025-01-03,2,111.4,113.0,0.0142,0.02,222.8
TXN-098F3B41DA19,BSK-421C4522CE09,STR-BBS-001,SKU-0259,PRD-0103,2025-01-03,2,46.62,47.0,0.008,0.0,93.24
TXN-9F90C06E5BD6,BSK-421C4522CE09,STR-BBS-001,SKU-0975,PRD-0390,2025-01-03,3,114.03,117.0,0.0254,0.03,342.09
TXN-F0FDD44BCF2D,BSK-421C4522CE09,STR-BBS-001,SKU-0724,PRD-0293,2025-01-03,2,121.46,127.0,0.0436,0.05,242.92
TXN-8C4B57354BB4,BSK-421C4522CE09,STR-BBS-001,SKU-0739,PRD-0299,2025-01-03,2,180.39,185.0,0.0249,0.04,360.78


## Creating silver.sales

**Purpose**: Create a comprehensive, fully denormalized sales fact table combining ALL facts and dimensions

**Source Tables**:
* `silver.transactions` (Transaction-level facts)
* `silver.line_items` (Line item-level facts with calculated discount_amount)
* `silver.customer_master` (Customer demographics)
* `silver.sku_master` (Complete product hierarchy)
* `silver.store_master` (Store location and attributes)

**What we're doing**:
* Joining transactions with line_items on transaction_id
* Enriching with customer_master to add customer demographics
* Enriching with sku_master to add complete product hierarchy
* Enriching with store_master to add store location/attributes
* Combining transaction-level context (payment, loyalty) with product-level details (SKU, quantity, price)
* Creating a single, wide table with complete context - ALL dimensions joined here

**Why this matters**:
* **One-stop table** for most common analyses - no joins needed
* Enables cross-analysis of customer behavior, product purchases, and transaction patterns
* Perfect for dashboards that need both transaction and product context
* Optimized for BI tools and reporting
* Simplifies queries for analysts - all context in one place
* **Single source of truth** - all dimension enrichment happens here only

**Grain**: One Row = One Line Item with full transaction, customer, product, and store context

In [0]:
# ===========================================================
# CREATE TABLE: silver.sales
# ===========================================================
# Purpose:
# Create a fully denormalized sales table that combines
# transaction facts, line item facts, and ALL dimension tables.
# This is the ultimate "wide table" for analytics.
#
# Grain:
# One Row = One Line Item with complete transaction, customer,
# product, and store context
#
# Source Tables:
# silver.transactions (transaction facts)
# silver.line_items (line item facts)
# silver.customer_master (customer demographics)
# silver.sku_master (product hierarchy)
# silver.store_master (store location/attributes)
#
# Transformations Applied:
# 1. Read transactions and line_items tables
# 2. INNER JOIN transactions with line_items on transaction_id
# 3. LEFT JOIN with customer_master on customer_id
# 4. LEFT JOIN with sku_master on sku_id
# 5. LEFT JOIN with store_master on store_id
# 6. Select all relevant columns with logical ordering
# 7. Save as Delta table
#
# Join Strategy:
# • INNER JOIN for transactions-line_items (every line item must have a transaction)
# • LEFT JOINs for dimensions (preserve all facts even if dimension lookup fails)
#
# Downstream Use Cases:
# • Single-table queries for most common analyses
# • Customer product preferences by demographics
# • Transaction patterns by product category
# • Payment method analysis by product type
# • Loyalty program effectiveness by product
# • Complete context for ML features
# • Dashboard reporting without joins
#
# Why create this table?
# Most analytical queries need transaction, customer, product,
# and store context together. This table provides everything
# in one place, eliminating complex multi-table joins.
# ===========================================================

from pyspark.sql.functions import col

# Read fact tables
df_transactions = spark.table('workspace.silver.transactions')
df_line_items = spark.table('workspace.silver.line_items')

# Read dimension tables
df_customer = spark.table('workspace.silver.customer_master')
df_sku = spark.table('workspace.silver.sku_master')
df_store = spark.table('workspace.silver.store_master')

# Join all tables
# Start with line_items, join with transactions, then enrich with all dimensions
df_sales_consolidated = df_line_items.alias('li') \
    .join(df_transactions.alias('txn'), 'transaction_id', 'inner') \
    .join(df_customer.alias('cust'), 'customer_id', 'left') \
    .join(df_sku.alias('sku'), 'sku_id', 'left') \
    .join(df_store.alias('store'), 'store_id', 'left') \
    .select(
        # IDs - Transactional
        'li.line_item_id',
        'txn.transaction_id',
        'txn.customer_id',
        'txn.store_id',
        # IDs - Material Hierarchy
        'li.sku_id',
        'li.product_id',
        'sku.category_id',
        'sku.department_id',
        # Date
        'txn.transaction_date',
        # Material Descriptors
        'sku.sku_name',
        'sku.product_name',
        'sku.brand',
        'sku.category_name',
        'sku.department_name',
        # Transaction-Level Facts
        'txn.payment_method',
        'txn.is_loyalty',
        'txn.total_amount',
        'txn.total_discount_amt',
        'txn.n_items',
        'txn.n_line_items',
        # Line Item-Level Facts
        'li.quantity',
        'li.amount',
        'li.mrp',
        'li.discount_pct',
        'li.discount_amount',  # Calculated field from line_items
        'li.line_total',
        # Customer Attributes
        'cust.gender',
        'cust.age_group',
        'cust.locality',
        'cust.loyalty_tier',
        # Product Attributes
        'sku.pack_size',
        'sku.pack_quantity',
        'sku.is_private_label',
        # Store Attributes
        'store.store_name',
        'store.city',
        'store.state',
        'store.region',
        'store.tier',
        'store.format'
    )

# Write to Silver layer as Delta table
# Use overwriteSchema to allow schema changes
df_sales_consolidated.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.silver.sales')

print("Successfully created silver.sales")

Successfully created silver.sales


In [0]:
# ===========================================================
# Verify Created Table
# ===========================================================
# Check sales table and validate the join
# ===========================================================

# Read the created table
df_sales_check = spark.table('workspace.silver.sales')

# Show statistics
print("Table Statistics:")
print(f"Total Records (Line Items): {df_sales_check.count():,}")
print(f"Unique Transactions: {df_sales_check.select('transaction_id').distinct().count():,}")
print(f"Unique Line Items: {df_sales_check.select('line_item_id').distinct().count():,}")
print(f"Unique Customers: {df_sales_check.select('customer_id').distinct().count():,}")
print(f"Unique SKUs: {df_sales_check.select('sku_id').distinct().count():,}")
print(f"Unique Stores: {df_sales_check.select('store_id').distinct().count():,}")

# Show sample records
print("\nSample records from silver.sales:")
display(df_sales_check.limit(10))

# Validate join quality
print("\n=== Join Quality Check ===")
print("This table should have the same row count as line_items (INNER JOIN)")
df_line_items_count = spark.table('workspace.silver.line_items').count()
print(f"Line Items count: {df_line_items_count:,}")
print(f"Consolidated count: {df_sales_check.count():,}")
print(f"Match: {df_line_items_count == df_sales_check.count()}")

Table Statistics:
Total Records (Line Items): 5,301,387
Unique Transactions: 1,380,771
Unique Line Items: 5,301,387
Unique Customers: 852,586
Unique SKUs: 1,604
Unique Stores: 21

Sample records from silver.sales:


line_item_id,transaction_id,customer_id,store_id,sku_id,product_id,category_id,department_id,transaction_date,sku_name,product_name,brand,category_name,department_name,payment_method,is_loyalty,total_amount,total_discount_amt,n_items,n_line_items,quantity,amount,mrp,discount_pct,discount_amount,line_total,gender,age_group,locality,loyalty_tier,pack_size,pack_quantity,is_private_label,store_name,city,state,region,tier,format
TXN-095D2C7B934E,BSK-BFFBDCA55057,ANON-5D97CCCA,STR-LKO-001,SKU-1015,PRD-0404,CAT-05-03,DEPT-05,2025-09-21,Bisleri Vedica Water 1L,Bisleri Vedica Water,Bisleri,Water & Soda,Beverages,Card,false,444.43,9.59,2,2,1,23.49,24.0,0.0214,0.01,23.49,null,null,null,null,1L,1.0,false,FreshMart Lucknow,Lucknow,Uttar Pradesh,North,2,Express
TXN-DC72FA83704E,BSK-BFFBDCA55057,ANON-5D97CCCA,STR-LKO-001,SKU-1065,PRD-0421,CAT-05-04,DEPT-05,2025-09-21,Protinex Health Health Drinks 500g,Protinex Health Health Drinks,Protinex,Health Drinks & Supplements,Beverages,Card,false,444.43,9.59,2,2,1,420.94,430.0,0.0211,0.09,420.94,null,null,null,null,500g,500.0,false,FreshMart Lucknow,Lucknow,Uttar Pradesh,North,2,Express
TXN-12BEE91D982E,BSK-E2F2A769C2C8,ANON-B88A6E05,STR-LKO-001,SKU-0905,PRD-0367,CAT-05-01,DEPT-05,2025-09-21,Brooke Bond Taj Tea 500g,Brooke Bond Taj Tea,Brooke Bond,Tea & Coffee,Beverages,Cash,false,599.26,13.74,3,3,1,321.25,328.0,0.0206,0.07,321.25,null,null,null,null,500g,500.0,false,FreshMart Lucknow,Lucknow,Uttar Pradesh,North,2,Express
TXN-28BBB1CA2113,BSK-E2F2A769C2C8,ANON-B88A6E05,STR-LKO-001,SKU-1115,PRD-0442,CAT-05-05,DEPT-05,2025-09-21,Sting Classic Energy 500ml,Sting Classic Energy,Sting,Energy & Sports Drinks,Beverages,Cash,false,599.26,13.74,3,3,1,77.35,79.0,0.0208,0.02,77.35,null,null,null,null,500ml,500.0,false,FreshMart Lucknow,Lucknow,Uttar Pradesh,North,2,Express
TXN-1582502FCE6C,BSK-E2F2A769C2C8,ANON-B88A6E05,STR-LKO-001,SKU-0936,PRD-0377,CAT-05-01,DEPT-05,2025-09-21,Bru Classic Tea 100g,Bru Classic Tea,Bru,Tea & Coffee,Beverages,Cash,false,599.26,13.74,3,3,1,200.66,206.0,0.0259,0.05,200.66,null,null,null,null,100g,100.0,false,FreshMart Lucknow,Lucknow,Uttar Pradesh,North,2,Express
TXN-8D55FE859460,BSK-178B9261AF68,ANON-2E773C59,STR-LKO-001,SKU-1133,PRD-0451,CAT-05-05,DEPT-05,2025-09-29,Sting Party Energy 500ml,Sting Party Energy,Sting,Energy & Sports Drinks,Beverages,UPI,false,693.21,15.8,3,3,1,79.94,81.0,0.0131,0.01,79.94,null,null,null,null,500ml,500.0,false,FreshMart Lucknow,Lucknow,Uttar Pradesh,North,2,Express
TXN-A5582019B8B4,BSK-178B9261AF68,ANON-2E773C59,STR-LKO-001,SKU-0902,PRD-0366,CAT-05-01,DEPT-05,2025-09-29,Tata Premium Tea 1kg,Tata Premium Tea,Tata,Tea & Coffee,Beverages,UPI,false,693.21,15.8,3,3,1,395.28,405.0,0.024,0.09,395.28,null,null,null,null,1kg,1.0,false,FreshMart Lucknow,Lucknow,Uttar Pradesh,North,2,Express
TXN-DF1999873C94,BSK-178B9261AF68,ANON-2E773C59,STR-LKO-001,SKU-1088,PRD-0430,CAT-05-04,DEPT-05,2025-09-29,Protinex Health Health Drinks 12 250g,Protinex Health Health Drinks 12,Protinex,Health Drinks & Supplements,Beverages,UPI,false,693.21,15.8,3,3,1,217.99,223.0,0.0225,0.05,217.99,null,null,null,null,250g,250.0,false,FreshMart Lucknow,Lucknow,Uttar Pradesh,North,2,Express
TXN-9AB64B6F36DF,BSK-FF72F0CE166E,CUST-LKO-0036,STR-LKO-001,SKU-0497,PRD-0202,CAT-02-05,DEPT-02,2025-10-10,Gits Fresh Frozen Snacks 300g,Gits Fresh Frozen Snacks,Gits,Frozen Snacks & Ready Meals,Dairy & Frozen,Cash,true,172.34,6.66,2,2,1,143.5,148.0,0.0304,0.04,143.5,M,18-25,Indira Nagar,Silver,300g,300.0,false,FreshMart Lucknow,Lucknow,Uttar Pradesh,North,2,Express
TXN-96E27266B0D0,BSK-FF72F0CE166E,CUST-LKO-0036,STR-LKO-001,SKU-0949,PRD-0381,CAT-05-02,DEPT-05,2025-10-10,Pepsi 250ml,Pepsi,Pepsi,Soft Drinks & Juices,Beverages,Cash,true,172.34,6.66,2,2,1,28.84,31.0,0.0697,0.02,28.84,M,18-25,Indira Nagar,Silver,250ml,250.0,false,FreshMart Lucknow,Lucknow,Uttar Pradesh,North,2,Express



=== Join Quality Check ===
This table should have the same row count as line_items (INNER JOIN)
Line Items count: 5,301,387
Consolidated count: 5,301,387
Match: True


## Creating silver.dim_date

**Purpose**: Create a reusable date dimension table for time-series analysis

**Source**: Generated programmatically based on transaction date range

**What we're doing**:
* Reading transaction dates directly from the CSV file
* Finding the min and max transaction dates
* Generating a complete date table spanning that range
* Adding calendar attributes (year, month, quarter, week, day)
* Including day of week and weekend flags
* Creating fiscal period attributes

**Why this matters**:
* Enables consistent date-based reporting across all dashboards
* Provides calendar intelligence (weekends, quarters, etc.)
* Fills gaps in sparse time-series data
* Standard date dimension pattern used across all Gold tables
* Essential for year-over-year and trend analysis

In [0]:
# ===========================================================
# Check Transaction Date Range from CSV
# ===========================================================
# Read transaction dates directly from the CSV file
# to determine what date range we need to cover
# ===========================================================

from pyspark.sql.functions import min, max, datediff, to_date

# Read transaction CSV file
df_trans_csv = spark.read.csv(
    '/Volumes/workspace/bronze/raw_data/fact_transaction.csv',
    header=True,
    inferSchema=True
)

# Convert to date if needed and get min/max dates
date_range = df_trans_csv.select(
    min(to_date('transaction_date')).alias('earliest_date'),
    max(to_date('transaction_date')).alias('latest_date')
).collect()[0]

print(f"Transaction Date Range:")
print(f"  Earliest: {date_range['earliest_date']}")
print(f"  Latest: {date_range['latest_date']}")
print(f"  Days to cover: {(date_range['latest_date'] - date_range['earliest_date']).days + 1}")

Transaction Date Range:
  Earliest: 2025-01-01
  Latest: 2026-05-31
  Days to cover: 516


In [0]:
# ===========================================================
# CREATE TABLE: silver.dim_date
# ===========================================================
# Purpose:
# Create a comprehensive date dimension table with calendar
# attributes for time-series analysis and reporting.
#
# Grain:
# One Row = One Calendar Date
#
# Source:
# Programmatically generated using PySpark based on
# transaction date range from the CSV file
#
# Date Range:
# Covers from earliest to latest transaction date
#
# Transformations Applied:
# 1. Get min/max dates from transaction CSV
# 2. Generate date range using date array
# 3. Extract calendar attributes (year, month, quarter, etc.)
# 4. Add day of week and weekend flag
# 5. Add fiscal period attributes
# 6. Save as Delta table
#
# Downstream Use Cases:
# • Time-series gap filling
# • Calendar-based filtering (weekdays, weekends, quarters)
# • Year-over-year comparisons
# • Trend analysis and forecasting
# • Seasonal pattern analysis
#
# Why create this table?
# Date dimensions are a standard pattern in dimensional modeling.
# They provide calendar intelligence and enable consistent
# time-based reporting across all business dashboards.
# ===========================================================

from pyspark.sql.functions import (
    col, explode, sequence, to_date, year, month, quarter, min, max,
    dayofmonth, dayofweek, weekofyear, dayofyear,
    date_format, when, expr
)

# Read transaction CSV to get date range
df_trans_csv = spark.read.csv(
    '/Volumes/workspace/bronze/raw_data/fact_transaction.csv',
    header=True,
    inferSchema=True
)

# Get min and max dates
date_range = df_trans_csv.select(
    min(to_date('transaction_date')).alias('min_date'),
    max(to_date('transaction_date')).alias('max_date')
)

# Generate date range using expr for proper interval type
df_date = date_range.select(
    explode(
        expr("sequence(min_date, max_date, interval 1 day)")
    ).alias('date')
)

# Add calendar attributes
df_dim_date = df_date.select(
    col('date').alias('date_key'),
    year('date').alias('year'),
    month('date').alias('month'),
    dayofmonth('date').alias('day_of_month'),
    # Formatted columns for reporting
    date_format('date', 'yyyyMM').alias('year_month'),  # 202501
    expr("concat('Q', quarter(date))").alias('quarter'),  # Q1, Q2, Q3, Q4
    expr("concat('W', lpad(weekofyear(date), 2, '0'))").alias('week'),  # W01, W02, ... W52
    # Week and day attributes
    weekofyear('date').alias('week_of_year'),
    dayofweek('date').alias('day_of_week'),
    dayofyear('date').alias('day_of_year'),
    # Name columns
    date_format('date', 'MMM').alias('month_short'),  # Jan, Feb, Mar
    date_format('date', 'MMMM').alias('month_name'),  # January, February
    date_format('date', 'EEE').alias('day_short'),  # Mon, Tue, Wed
    date_format('date', 'EEEE').alias('day_name'),  # Monday, Tuesday
    # Flags
    when(dayofweek('date').isin([1, 7]), True).otherwise(False).alias('is_weekend'),
    when(dayofweek('date').isin([2, 3, 4, 5, 6]), True).otherwise(False).alias('is_weekday')
)

# Write to Silver layer as Delta table
# Use overwriteSchema to allow column changes (quarter changed from int to string)
df_dim_date.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.silver.dim_date')

print("Successfully created silver.dim_date")

Successfully created silver.dim_date


In [0]:
# ===========================================================
# Verify Created Table
# ===========================================================
# Check date dimension and validate coverage
# ===========================================================

# Read the created table
df_date_check = spark.table('workspace.silver.dim_date')

# Show statistics
from pyspark.sql.functions import sum as spark_sum, when, col

stats = df_date_check.select(
    spark_sum(when(col('is_weekend'), 1).otherwise(0)).alias('weekend_days'),
    spark_sum(when(col('is_weekday'), 1).otherwise(0)).alias('weekday_days')
).collect()[0]

print("Table Statistics:")
print(f"Total Dates: {df_date_check.count():,}")
print(f"Unique Years: {df_date_check.select('year').distinct().count():,}")
print(f"Weekend Days: {stats['weekend_days']:,}")
print(f"Weekday Days: {stats['weekday_days']:,}")

# Show sample records
print("\nSample records from silver.dim_date:")
display(df_date_check.orderBy('date_key').limit(10))

Table Statistics:
Total Dates: 516
Unique Years: 2
Weekend Days: 148
Weekday Days: 368

Sample records from silver.dim_date:


date_key,year,month,day_of_month,year_month,quarter,week,week_of_year,day_of_week,day_of_year,month_short,month_name,day_short,day_name,is_weekend,is_weekday
2025-01-01,2025,1,1,202501,Q1,W01,1,4,1,Jan,January,Wed,Wednesday,false,true
2025-01-02,2025,1,2,202501,Q1,W01,1,5,2,Jan,January,Thu,Thursday,false,true
2025-01-03,2025,1,3,202501,Q1,W01,1,6,3,Jan,January,Fri,Friday,false,true
2025-01-04,2025,1,4,202501,Q1,W01,1,7,4,Jan,January,Sat,Saturday,true,false
2025-01-05,2025,1,5,202501,Q1,W01,1,1,5,Jan,January,Sun,Sunday,true,false
2025-01-06,2025,1,6,202501,Q1,W02,2,2,6,Jan,January,Mon,Monday,false,true
2025-01-07,2025,1,7,202501,Q1,W02,2,3,7,Jan,January,Tue,Tuesday,false,true
2025-01-08,2025,1,8,202501,Q1,W02,2,4,8,Jan,January,Wed,Wednesday,false,true
2025-01-09,2025,1,9,202501,Q1,W02,2,5,9,Jan,January,Thu,Thursday,false,true
2025-01-10,2025,1,10,202501,Q1,W02,2,6,10,Jan,January,Fri,Friday,false,true


## Silver Layer Creation Complete!

**Summary**: All Silver layer tables have been successfully created.

### Tables Created:

1. **silver.customer_master** - Customer dimension with demographics and loyalty attributes
2. **silver.store_master** - Store dimension with location and operational attributes
3. **silver.sku_master** - Complete product hierarchy (SKU → Product → Category → Department)
4. **silver.transactions** - Pure transaction-level fact table (no dimension joins)
5. **silver.line_items** - Pure line item-level fact table with calculated discount_amount (no dimension joins)
6. **silver.sales** - Fully denormalized sales table combining ALL facts and dimensions (single source of truth)
7. **silver.dim_date** - Date dimension with reporting-friendly calendar attributes

### Architecture Highlights:

**Fact Tables (Facts Only)**:
* `transactions` and `line_items` contain only transaction/line item facts
* No dimension joins - keeps them clean and flexible

**Dimension Tables**:
* `customer_master`, `store_master`, `sku_master` provide lookup attributes
* Can be used independently for dimension analysis

**Consolidated Sales Table**:
* `sales` is where ALL dimension enrichment happens
* Combines transactions + line_items + all dimensions
* Single table for most analytical queries
* Includes calculated metric: `discount_amount`

### Next Steps:

* **Explore the data**: Run EDA notebook on `silver.sales`
* **Create Gold layer**: Build aggregated metrics and business KPIs
* **Build dashboards**: Connect BI tools to `silver.sales` for easy querying
* **Data quality**: Document any data quality findings and decisions  